<a href="https://colab.research.google.com/github/Ali-Hamza-developer/flyrank-ml-internship/blob/main/%5Cwork%5Cnotebooks%5Cw05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane
**Lane: Refresh / Content Opportunity Scoring**

Run top to bottom (Runtime → Run all). Requires `HF_TOKEN` Colab Secret, same as w03-w07.

In [1]:
!pip install -q duckdb scikit-learn

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month — matches w03/w05/w06/w07, never the sealed _sample
print('DuckDB ready, target month =', MONTH)

DuckDB ready, target month = 2026-03


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The w05 leakage-hunt notebook already established an honest baseline of **AUC = 0.881** using a plain logistic regression on five prior-window-only features (`log_impressions_90d`, `avg_position_90d`, `ctr_90d`, `days_since_last_update`, `word_count`). That number is the one to beat here, on the same leakage-safe feature construction.

For the capstone model I'm choosing a **gradient-boosted tree ensemble** (`HistGradientBoostingClassifier`) instead of a more expressive linear model, for three reasons tied directly to what earlier weeks found in this data:

1. **Tiered, non-additive relationships.** The w04 rule baseline had to bucket `avg_position_90d` into position tiers before comparing CTR within each tier — comparing raw CTR across tiers directly was flagged as a named mistake in the lane guide. A tree-based model can learn tier-like splits on `avg_position_90d` natively instead of relying on a single linear coefficient that assumes a straight-line relationship across the whole range.
2. **Missing data without manual imputation choices.** The w05 audit found substantial missingness in `word_count` (~54,711 of 136,974 rows) and `engagement_rate_90d` (~96,657 rows). `HistGradientBoostingClassifier` handles missing values natively during split-finding, which avoids introducing another imputation decision on top of the ones already made and documented in w05 (`ctr_90d` filled with 0.0, `engagement_rate_90d` filled with -1).
3. **Interpretability retained.** Permutation importance and direct error inspection (Section 4) still work on a tree ensemble, so the interpretability the internship deliverables need isn't traded away for the non-linearity gain.

The logistic regression from w05 is **not discarded** — it's re-run in this notebook on the exact same feature set, but under this notebook's own grouped train/test split (Section 2), so the comparison in Section 3 is apples-to-apples rather than comparing across two different splitting protocols.

In [2]:
# Rebuild the honest feature base — same construction as w03/w05/w06/w07, no future-window or product-flag columns
base = con.sql(f"""
    WITH window_90d AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_90d,
            SUM(gsc_clicks) AS clicks_90d,
            AVG(gsc_avg_position) AS avg_position_90d,
            SUM(ga4_sessions) AS sessions_90d,
            SUM(ga4_engaged_sessions) AS engaged_sessions_90d
        FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        w.client_hash_id,
        w.content_hash_id,
        LOG(1 + w.impressions_90d) AS log_impressions_90d,
        w.avg_position_90d,
        CASE WHEN w.impressions_90d > 0 THEN w.clicks_90d::DOUBLE / w.impressions_90d ELSE NULL END AS ctr_90d,
        CASE WHEN w.sessions_90d > 0 THEN w.engaged_sessions_90d::DOUBLE / w.sessions_90d ELSE NULL END AS engagement_rate_90d,
        (DATE '2026-03-31' - COALESCE(d.last_optimized_date, d.content_created_date)) AS days_since_last_update,
        d.word_count,
        d.content_type,
        d.main_intent,
        d.competition_level
    FROM window_90d w
    JOIN read_parquet('{REL}/dim_content.parquet') d
      ON w.content_hash_id = d.content_hash_id AND w.client_hash_id = d.client_hash_id
    WHERE w.impressions_90d > 0
      AND (DATE '2026-03-31' - COALESCE(d.last_optimized_date, d.content_created_date)) >= 0
""").df()

print('rows:', len(base))
print('distinct clients:', base['client_hash_id'].nunique())
print('\nmissing values per column:')
print(base.isna().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 136974
distinct clients: 47

missing values per column:
client_hash_id                0
content_hash_id               0
log_impressions_90d           0
avg_position_90d              0
ctr_90d                       0
engagement_rate_90d       96657
days_since_last_update        0
word_count                54711
content_type                  0
main_intent               15746
competition_level         16372
dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

This split is **grouped by `client_hash_id`**, not a plain random row split. Content items belonging to the same client share client-level effects — CMS/template quirks, editorial cadence, niche — that a random row split would let leak between train and test (the model could partly memorize a client's baseline behavior instead of learning general prior-window-to-outcome signal). Grouping by client means every content item from a given client lands entirely in train or entirely in test, so the held-out score reflects performance on **clients the model has never seen**, which matches how this pipeline would actually be used going forward (scoring content for clients not in the training slice).

It is **not additionally time-aware** at the row level, because every row in `base` comes from the same feature snapshot (`month=2026-03`). Time-awareness is instead enforced the same way it was in w03/w05/w06/w07: the label (`future_decline`) is built from a **separate, later month partition** (`month=2026-04`), so the outcome window never overlaps the feature window regardless of how train/test rows are split.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

# Label: same 30-day forward window and decline definition as w03/w05/w06
label_window = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_next30
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    WHERE report_date <= DATE '2026-04-30'
    GROUP BY 1, 2
""").df()

modeling_df = base.merge(label_window, on=['client_hash_id', 'content_hash_id'], how='inner')
modeling_df['future_decline'] = (
    modeling_df['impressions_next30'] < 0.7 * (2 ** modeling_df['log_impressions_90d'] - 1)
).astype(int)

# The label-window column must never reach the feature matrix — drop it immediately after building the label
modeling_df = modeling_df.drop(columns=['impressions_next30'])

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(modeling_df, groups=modeling_df['client_hash_id']))
train_df = modeling_df.iloc[train_idx].copy()
test_df = modeling_df.iloc[test_idx].copy()

print('train rows:', len(train_df), '| train clients:', train_df['client_hash_id'].nunique())
print('test rows:', len(test_df), '| test clients:', test_df['client_hash_id'].nunique())
print('client overlap between train and test (should be empty set):',
      set(train_df['client_hash_id']).intersection(set(test_df['client_hash_id'])))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train rows: 103540 | train clients: 35
test rows: 33433 | test clients: 12
client overlap between train and test (should be empty set): set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Two models are trained and scored on the **same grouped split** and the **same two metrics**: AUC (used throughout w03/w05/w06 as the leakage-sanity metric) and **Precision@50** (the success metric chosen in w02, since it's the number that actually reflects how an SEO analyst will use the ranked queue).

- **Logistic regression** — the exact five honest features from w05 (`log_impressions_90d`, `avg_position_90d`, `ctr_90d`, `days_since_last_update`, `word_count`), re-run on this notebook's grouped split so it's a fair comparison point rather than reusing w05's differently-split number directly.
- **Gradient boosted trees** — the same five honest features plus the three static content-context categoricals (`content_type`, `main_intent`, `competition_level`), which the tree model can consume natively without one-hot encoding.

w05's originally reported AUC (0.881, different split protocol) is kept in the table only as a reference point, not as a like-for-like row.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

features_honest = ['log_impressions_90d', 'avg_position_90d', 'ctr_90d',
                    'days_since_last_update', 'word_count']
features_context = features_honest + ['content_type', 'main_intent', 'competition_level']

def precision_at_k(y_true, y_score, k=50):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    order = np.argsort(-y_score)
    top_k = order[:k]
    return y_true[top_k].mean()

# --- Logistic regression: same honest feature set as w05, this notebook's grouped split ---
lr_train = train_df.dropna(subset=features_honest + ['future_decline'])
lr_test = test_df.dropna(subset=features_honest + ['future_decline'])

clf_lr = LogisticRegression(max_iter=1000)
clf_lr.fit(lr_train[features_honest], lr_train['future_decline'])
lr_scores = clf_lr.predict_proba(lr_test[features_honest])[:, 1]
lr_auc = roc_auc_score(lr_test['future_decline'], lr_scores)
lr_p50 = precision_at_k(lr_test['future_decline'], lr_scores, k=50)

# --- Gradient boosted trees: honest features + context categoricals, native NaN + categorical handling ---
gb_train = train_df.copy()
gb_test = test_df.copy()
for col in ['content_type', 'main_intent', 'competition_level']:
    gb_train[col] = gb_train[col].astype('category')
    gb_test[col] = gb_test[col].astype(pd.CategoricalDtype(categories=gb_train[col].cat.categories))

categorical_idx = [features_context.index(c) for c in ['content_type', 'main_intent', 'competition_level']]
clf_gb = HistGradientBoostingClassifier(categorical_features=categorical_idx, random_state=42)
clf_gb.fit(gb_train[features_context], gb_train['future_decline'])
gb_scores = clf_gb.predict_proba(gb_test[features_context])[:, 1]
gb_auc = roc_auc_score(gb_test['future_decline'], gb_scores)
gb_p50 = precision_at_k(gb_test['future_decline'], gb_scores, k=50)

comparison = pd.DataFrame({
    'model': [
        'w05 honest baseline (reported, different split)',
        'logistic regression (this grouped split)',
        'gradient boosted trees (this grouped split)',
    ],
    'AUC': [0.881, round(lr_auc, 3), round(gb_auc, 3)],
    'Precision@50': ['n/a — not computed in w05', round(lr_p50, 3), round(gb_p50, 3)],
})
print(comparison.to_string(index=False))

print('\nColumns used — logistic regression:', features_honest)
print('Columns used — gradient boosted trees:', features_context)
print('label-window column (impressions_next30) excluded from both feature lists:',
      'impressions_next30' not in features_context)

                                          model   AUC              Precision@50
w05 honest baseline (reported, different split) 0.881 n/a — not computed in w05
       logistic regression (this grouped split) 0.837                      0.62
    gradient boosted trees (this grouped split) 0.887                      0.52

Columns used — logistic regression: ['log_impressions_90d', 'avg_position_90d', 'ctr_90d', 'days_since_last_update', 'word_count']
Columns used — gradient boosted trees: ['log_impressions_90d', 'avg_position_90d', 'ctr_90d', 'days_since_last_update', 'word_count', 'content_type', 'main_intent', 'competition_level']
label-window column (impressions_next30) excluded from both feature lists: True


**Fill in after running:** which model wins on **Precision@50** — that number matters more here than AUC, since it mirrors how the SEO analyst actually consumes the ranked queue (they review a fixed-size list, not a probability curve). State the actual AUC and Precision@50 values from the table above, and note whether the gradient-boosted model's gain (if any) over the grouped-split logistic regression is large enough to justify the added complexity, or whether it's within noise given the number of distinct test clients printed in Section 2.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Permutation importance is used instead of a native split-gain importance, since permutation importance is measured directly against the held-out AUC and isn't biased toward high-cardinality features the way impurity-based importances can be. Error rows are pulled at a 0.5 probability threshold on the gradient-boosted model's test predictions — this threshold is a starting point for inspection, not a claim about the right operating point for deployment.

In [5]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    clf_gb, gb_test[features_context], gb_test['future_decline'],
    n_repeats=10, random_state=42, scoring='roc_auc'
)
importance_table = pd.DataFrame({
    'feature': features_context,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)
print('Permutation importance (drop in AUC when feature is shuffled):')
print(importance_table.to_string(index=False))

# Error inspection: highest-confidence false positives and false negatives
gb_test_scored = gb_test.copy()
gb_test_scored['predicted_prob'] = gb_scores
gb_test_scored['predicted_label'] = (gb_test_scored['predicted_prob'] >= 0.5).astype(int)

false_positives = gb_test_scored[
    (gb_test_scored['predicted_label'] == 1) & (gb_test_scored['future_decline'] == 0)
].sort_values('predicted_prob', ascending=False).head(10)

false_negatives = gb_test_scored[
    (gb_test_scored['predicted_label'] == 0) & (gb_test_scored['future_decline'] == 1)
].sort_values('predicted_prob').head(10)

cols_to_show = features_context + ['predicted_prob', 'future_decline']
print('\nTop false positives (model flagged decline, page actually stayed stable):')
print(false_positives[cols_to_show].to_string(index=False))
print('\nTop false negatives (model missed an actual decline):')
print(false_negatives[cols_to_show].to_string(index=False))

Permutation importance (drop in AUC when feature is shuffled):
               feature  importance_mean  importance_std
   log_impressions_90d         0.298463        0.004143
            word_count         0.023036        0.000815
      avg_position_90d         0.008942        0.000495
days_since_last_update         0.002871        0.001521
               ctr_90d         0.001906        0.000248
           main_intent         0.001154        0.000194
     competition_level         0.000896        0.000165
          content_type         0.000089        0.000017

Top false positives (model flagged decline, page actually stayed stable):
 log_impressions_90d  avg_position_90d  ctr_90d  days_since_last_update  word_count    content_type   main_intent competition_level  predicted_prob  future_decline
             0.30103               0.0      0.0                      74        4659 keyword article    commercial               LOW        0.820716               0
             0.30103          

**Fill in after running, 2-3 sentences, observational language only ("observed", "appears to", never "proves"):**
- Which feature(s) dominate the permutation importance table — does the ranking match the flag-linked position-vs-CTR signal from w06, or does something else (e.g. `days_since_last_update`) dominate instead?
- Looking at the false-positive and false-negative rows above, is there a shared pattern (e.g. very new content with thin impression history, or high-competition pages with unstable CTR) that explains where this model is wrong?

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.